# NOAA GHCN-D — Verificación de estructura y faltantes

Se selecciona automáticamente una estación con registros reales para la variable y período parametrizados; se verifica la estructura y la presencia/ausencia de observaciones. El CSV temporal se elimina al final.


In [27]:
%pip -q install boto3

import csv
import gzip
import io
import shutil
from collections import Counter
from pathlib import Path
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from datetime import datetime, date, timedelta


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
# Parámetros reutilizables de la prueba
codigo_pais = "NZ"
variable_control = "PRCP"
fecha_desde = "20160101"
fecha_hasta = "20261231"

In [24]:
BUCKET = "noaa-ghcn-pds"
PREFIJO = f"csv.gz/by_station/{codigo_pais.upper()}"

s3 = boto3.client("s3", region_name="us-east-1", config=Config(signature_version=UNSIGNED))
paginador = s3.get_paginator("list_objects_v2")
claves = sorted(
    obj["Key"]
    for pagina in paginador.paginate(Bucket=BUCKET, Prefix=PREFIJO)
    for obj in pagina.get("Contents", [])
    if obj["Key"].endswith(".csv.gz")
)
if not claves:
    raise ValueError(f"No hay archivos para el país {codigo_pais}")

# Elegir de manera reproducible el primer archivo que realmente contenga
# la variable en el período solicitado (no basta con figurar en el inventario).
clave = None
contenido = None
for candidata in claves:
    respuesta = s3.get_object(Bucket=BUCKET, Key=candidata)
    try:
        comprimido_bytes = respuesta["Body"].read()
    finally:
        respuesta["Body"].close()
    with gzip.open(io.BytesIO(comprimido_bytes), mode="rt", encoding="utf-8") as origen:
        tiene_variable = any(
            len(fila) == 8 and fila[2] == variable_control
            and fecha_desde <= fila[1] <= fecha_hasta
            for fila in csv.reader(origen)
        )
    if tiene_variable:
        clave, contenido = candidata, comprimido_bytes
        break

if clave is None:
    raise ValueError(f"Ninguna estación de {codigo_pais} tiene registros de {variable_control} en el período")

archivo = Path(clave).name.removesuffix(".gz")
print("Estación seleccionada:", archivo.removesuffix(".csv"))
print("Objeto público:", clave)
with gzip.open(io.BytesIO(contenido), "rb") as comprimido:
    with open(archivo, "wb") as salida:
        shutil.copyfileobj(comprimido, salida)
archivo = Path(archivo)
print("CSV descargado y descomprimido:", archivo)


Estación seleccionada: NZ000093292
Objeto público: csv.gz/by_station/NZ000093292.csv.gz
CSV descargado y descomprimido: NZ000093292.csv


## Inspección del formato original

Se leen los campos **como texto**, sin convertir fechas, unidades ni indicadores. Se muestra el primer registro completo y se comprueba si hay encabezado. El conteo de columnas se realiza sobre todo el archivo.

In [25]:
recuento_columnas = Counter()
ejemplos = []
filas_vacias = 0
filas_totales = 0

with archivo.open("r", encoding="utf-8-sig", newline="") as f:
    lector = csv.reader(f)
    for fila in lector:
        if not fila or all(not x.strip() for x in fila):
            filas_vacias += 1
            continue
        filas_totales += 1
        recuento_columnas[len(fila)] += 1
        if len(ejemplos) < 6:
            ejemplos.append(fila)

if not ejemplos:
    raise ValueError("El CSV no contiene registros")

print("Cantidad de filas no vacías (incluye encabezado, si existe):", filas_totales)
print("Filas vacías:", filas_vacias)
print("Cantidad de columnas por fila:", dict(sorted(recuento_columnas.items())))
print("Primera fila (lista completa):", repr(ejemplos[0]))
print("Siguientes filas de ejemplo:")
for fila in ejemplos[1:]:
    print(repr(fila))

primera = [v.strip().upper() for v in ejemplos[0]]
parece_encabezado = ("DATE" in primera or "FECHA" in primera) and ("ELEMENT" in primera or "VARIABLE" in primera)
print("¿La primera fila parece encabezado?:", parece_encabezado)
print("Estructura uniforme:", len(recuento_columnas) == 1)
if parece_encabezado:
    print("Nombres de columnas:", ejemplos[0])
else:
    print("ATENCIÓN: no se detectó un encabezado reconocible; verificar el formato antes de continuar")

Cantidad de filas no vacías (incluye encabezado, si existe): 65445
Filas vacías: 0
Cantidad de columnas por fila: {8: 65445}
Primera fila (lista completa): ['NZ000093292', '19620201', 'TMAX', '209', '', '', 'G', '']
Siguientes filas de ejemplo:
['NZ000093292', '19620202', 'TMAX', '227', '', '', 'G', '']
['NZ000093292', '19620203', 'TMAX', '266', '', '', 'G', '']
['NZ000093292', '19620204', 'TMAX', '256', '', '', 'G', '']
['NZ000093292', '19620205', 'TMAX', '263', '', '', 'G', '']
['NZ000093292', '19620206', 'TMAX', '183', '', '', 'G', '']
¿La primera fila parece encabezado?: False
Estructura uniforme: True
ATENCIÓN: no se detectó un encabezado reconocible; verificar el formato antes de continuar


# Estructura oficial del CSV por estación de NOAA GHCN-D

In [26]:
COLUMNAS_NOAA = [
    "STATION",
    "DATE",
    "ELEMENT",
    "DATA_VALUE",
    "M_FLAG",
    "Q_FLAG",
    "S_FLAG",
    "OBS_TIME"
]

estructura_valida = (
    set(recuento_columnas.keys()) == {len(COLUMNAS_NOAA)}
    and not parece_encabezado
    and filas_vacias == 0
)

print("Columnas esperadas:", COLUMNAS_NOAA)
print("Cantidad esperada:", len(COLUMNAS_NOAA))
print("Cantidad observada:", dict(recuento_columnas))
print("CSV sin encabezado:", not parece_encabezado)
print("Estructura válida:", estructura_valida)

if not estructura_valida:
    raise ValueError("El archivo no cumple la estructura esperada")

Columnas esperadas: ['STATION', 'DATE', 'ELEMENT', 'DATA_VALUE', 'M_FLAG', 'Q_FLAG', 'S_FLAG', 'OBS_TIME']
Cantidad esperada: 8
Cantidad observada: {8: 65445}
CSV sin encabezado: True
Estructura válida: True


## Control de faltantes de la variable seleccionada

La ausencia de una fila para una fecha se distingue de los valores especiales, los campos vacíos y las mediciones observadas iguales a cero. Los indicadores `Q_FLAG` se cuentan por separado: una observación con indicador de calidad no se considera válida para calcular cobertura. Se controla el período indicado en los parámetros y se muestra una tabla por año. La última celda borra el CSV local.


In [28]:
inicio = datetime.strptime(fecha_desde, "%Y%m%d").date()
fin = datetime.strptime(fecha_hasta, "%Y%m%d").date()
if inicio > fin:
    raise ValueError("La fecha inicial es posterior a la final")

# Una fecha futura no debe clasificarse como observación ausente.
fin_analisis = min(fin, date.today())
if fin_analisis < inicio:
    raise ValueError("El período todavía no comenzó")

registros_variable = 0
faltantes_9999 = 0
vacios = 0
no_numericos = 0
qflag_informado = 0
ceros_validos = 0
fechas_con_registro = set()
fechas_validas = set()
conteo_fechas = Counter()

with archivo.open("r", encoding="utf-8-sig", newline="") as f:
    for fila in csv.reader(f):
        if len(fila) != 8:
            raise ValueError(f"Fila con {len(fila)} columnas: {fila[:3]}")
        estacion, fecha_txt, variable, valor_txt, mf, qf, sf, hora = fila
        if variable != variable_control or not fecha_desde <= fecha_txt <= fecha_hasta:
            continue
        fecha = datetime.strptime(fecha_txt, "%Y%m%d").date()
        if fecha > fin_analisis:
            continue
        registros_variable += 1
        fechas_con_registro.add(fecha)
        conteo_fechas[fecha] += 1
        valor = valor_txt.strip()
        if qf.strip():
            qflag_informado += 1
        if valor == "-9999":
            faltantes_9999 += 1
        elif not valor:
            vacios += 1
        else:
            try:
                numero = int(valor)
            except ValueError:
                no_numericos += 1
            else:
                if not qf.strip():
                    fechas_validas.add(fecha)
                    if numero == 0:
                        ceros_validos += 1

print(f"Estación: {archivo.stem} | Variable: {variable_control}")
print(f"Período evaluado: {inicio} a {fin_analisis}")
print(f"Registros: {registros_variable:,} | -9999: {faltantes_9999:,} | "
      f"Vacíos: {vacios:,} | No numéricos: {no_numericos:,}")
print(f"Con Q_FLAG: {qflag_informado:,} | Ceros válidos: {ceros_validos:,} | "
      f"Fechas duplicadas: {sum(n > 1 for n in conteo_fechas.values()):,}")
print(f"{'Año':<6} {'Días':>6} {'Con registro':>13} {'Válidos':>9} {'Sin registro':>13} {'Cobertura %':>12}")
for anio in range(inicio.year, fin_analisis.year + 1):
    a = max(inicio, date(anio, 1, 1))
    b = min(fin_analisis, date(anio, 12, 31))
    dias = (b-a).days + 1
    con_registro = sum(a <= d <= b for d in fechas_con_registro)
    validos = sum(a <= d <= b for d in fechas_validas)
    print(f"{anio:<6} {dias:>6} {con_registro:>13} {validos:>9} "
          f"{dias-con_registro:>13} {100*validos/dias:>11.2f}%")


Estación: NZ000093292 | Variable: PRCP
Período evaluado: 2016-01-01 a 2026-10-30
Registros: 3,397 | -9999: 0 | Vacíos: 0 | No numéricos: 0
Con Q_FLAG: 0 | Ceros válidos: 2,019 | Fechas duplicadas: 0
Año      Días  Con registro   Válidos  Sin registro  Cobertura %
2016      366           360       360             6       98.36%
2017      365           361       361             4       98.90%
2018      365           352       352            13       96.44%
2019      365           354       354            11       96.99%
2020      366           364       364             2       99.45%
2021      365           304       304            61       83.29%
2022      365           364       364             1       99.73%
2023      365           355       355            10       97.26%
2024      366           350       350            16       95.63%
2025      365           233       233           132       63.84%
2026      303             0         0           303        0.00%


In [29]:
# Eliminar únicamente el CSV utilizado en la verificación
if archivo.is_file():
    archivo.unlink()
    print(f"Archivo eliminado: {archivo}")
else:
    print("El archivo ya no existe.")

Archivo eliminado: NZ000093292.csv
